In [0]:
%sql
CREATE OR REPLACE VIEW retail_q.retail_semantic.retail_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: retail_q.retail_gold.fact_sales
comment: Retail sales metrics covering revenue, discounts, quantity, transactions, and average deal size across product, customer, and time dimensions.
joins:
  - name: products
    source: retail_q.retail_gold.dim_product
    on: source.product_id = products.product_id
    cardinality: many_to_one
  - name: customers
    source: retail_q.retail_gold.dim_customer
    on: source.customer_id = customers.customer_id
    cardinality: many_to_one
  - name: calendar
    source: retail_q.retail_gold.dim_calendar
    on: source.transaction_date = calendar.date
    cardinality: many_to_one
dimensions:
  - name: Product Category
    expr: products.category
    display_name: Product Category
    comment: Category of the product sold (e.g. Electronics, Apparel)
    synonyms:
      - product type
      - category
  - name: Customer Industry
    expr: customers.industry
    display_name: Customer Industry
    comment: Industry sector of the purchasing customer
    synonyms:
      - industry
      - sector
  - name: Sales Channel
    expr: source.sales_channel
    display_name: Sales Channel
    comment: Channel through which the sale was made (e.g. Online, In-store)
    synonyms:
      - channel
  - name: Payment Mode
    expr: source.payment_mode
    display_name: Payment Mode
    comment: Method of payment used for the transaction (e.g. Credit Card, Cash)
    synonyms:
      - payment method
      - payment type
  - name: Transaction Month
    expr: DATE_TRUNC('MONTH', source.transaction_date)
    display_name: Transaction Month
    comment: Calendar month in which the transaction occurred
    format:
      type: date
      date_format: locale_short_month
    synonyms:
      - month
      - sales month
  - name: Customer Type
    expr: customers.customer_type
    display_name: Customer Type
    comment: Segment or category of the customer (e.g. Enterprise, Small Business)
    synonyms:
      - customer segment
      - account type
  - name: Customer Name
    expr: customers.customer_name
    display_name: Customer Name
    comment: Name of the purchasing customer or account
    synonyms:
      - account name
      - customer
  - name: Billing City
    expr: customers.billing_city
    display_name: Billing City
    comment: City from the customer's billing address
    synonyms:
      - city
  - name: Billing State
    expr: customers.billing_state
    display_name: Billing State
    comment: State from the customer's billing address
    synonyms:
      - state
      - region
  - name: Billing Country
    expr: customers.billing_country
    display_name: Billing Country
    comment: Country from the customer's billing address
    synonyms:
      - country
measures:
  - name: Total Revenue
    expr: SUM(source.selling_price * source.quantity)
    display_name: Total Revenue
    comment: Total revenue from all sales (selling price multiplied by quantity)
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - revenue
      - sales
      - gross sales
  - name: Total Discount
    expr: SUM(source.discount_amount)
    display_name: Total Discount
    comment: Total discount amount applied across all transactions
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - discount
      - discounts
  - name: Total Quantity Sold
    expr: SUM(source.quantity)
    display_name: Total Quantity Sold
    comment: Total number of units sold across all transactions
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - quantity
      - units sold
      - volume
  - name: Transaction Count
    expr: COUNT(1)
    display_name: Transaction Count
    comment: Number of sales transactions
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - transactions
      - order count
      - number of transactions
  - name: Avg Deal Size
    expr: AVG(source.amount)
    display_name: Avg Deal Size
    comment: Average deal or opportunity amount per transaction
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - average deal
      - avg opportunity
      - average order value
  - name: Unique Customers
    expr: COUNT(DISTINCT source.customer_id)
    display_name: Unique Customers
    comment: Number of distinct customers who made purchases
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - distinct customers
      - customer count
      - accounts
$$